# 01. In-context learning (ICL)

Classifying word senses into supersenses by **prompting** `Qwen2.5-7B-Instruct` (4-bit), without training.
All choices are tuned on **`rand-dev`**, changing **one factor at a time** (see `docs/prompt_literature.md`):

1. prompt language: English instruction and labels vs everything in French;
2. number of demonstrations: 0 / 5 / 25 / 50;
3. demonstration selection: random vs balanced over classes vs most similar definitions;
4. with vs without the lemma.

The best configuration is run once on `rand-test` and `freq-test` at the end of the notebook.
Runtime: GPU (T4), on Colab or on Kaggle (*Save & Run All* runs it in the background).

In [1]:
# Setup: on Colab or Kaggle, clone (or update) the repository so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/Alhassliebe/LAB1_classification_with_llms.git"
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/LAB1_classification_with_llms"
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
!pip install -q bitsandbytes  # 4-bit weights; the rest is preinstalled in Colab and Kaggle

In [3]:
# All hyperparameters of this notebook.
CONFIG = {
    "model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # see docs/model_choice.md
    "tuning_split": "rand-dev",
    "max_new_tokens": 10,  # the longest label is 5 tokens; a longer answer is cut and parsed anyway
    "seed": 42,
}

In [4]:
import copy
import time

import pandas as pd
import torch
from tqdm.auto import tqdm
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data import HYPERSENSES, SUPER2HYPER, SUPERSENSES, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, log_experiment, paired_bootstrap, save_predictions
from src.label_scoring import label_token_ids, score_labels
from src.prompts import ShotSelector, build_prompt, parse_answer

# Hides a warning printed at every generate() call (the model config sets max_length, we set max_new_tokens).
transformers.logging.set_verbosity_error()
df = load_data()
train = get_split(df, "train")
dev = get_split(df, CONFIG["tuning_split"])

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], device_map="auto", dtype=torch.float16)
print(f"GPU memory used by the weights: {torch.cuda.memory_allocated() / 1e9:.1f} GB" if torch.cuda.is_available() else "CPU")

GPU memory used by the weights: 5.5 GB


## Prediction

Two decoding modes. **Generation**: the model answers freely (greedy decoding) and `parse_answer` maps the text to a
label. **Label scoring** (`logprob`): the model scores the 25 possible answers (label + end of turn) and we take the most
probable one, so every prediction is a valid label (`src/label_scoring.py`).
The beginning of the prompt is **the same for every query**: the instruction, plus the demonstrations with `random` or
`by_class` selection (with `similar` selection the demonstrations change for each query). We encode this prefix once and
reuse its key/value cache, so each query only costs its own tokens (less GPU time and energy; compare the
`runtime_min` column of the summary).

In [6]:
def encode(messages, generation_prompt):
    """Token ids of a chat, as the model sees it."""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=generation_prompt, tokenize=False)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)


@torch.no_grad()
def predict(eval_df, selector, lang, with_lemma, decoding):
    """Predicted supersense and raw output (generated text, or the 3 best labels with their log-prob) per row."""
    labels_ids = label_token_ids(tokenizer, lang)
    prefix_cache = None
    preds, raws = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        messages = build_prompt(row, selector(row), lang, with_lemma)
        input_ids = encode(messages, generation_prompt=True)
        if prefix_cache is None:  # first query: encode the shared prefix once
            shared = messages[:1] if selector.strategy == "similar" else messages[:-1]
            prefix_ids = encode(shared, generation_prompt=False)
            prefix_cache = model(prefix_ids, use_cache=True).past_key_values
        assert torch.equal(input_ids[0, : prefix_ids.shape[1]], prefix_ids[0])
        cache = copy.deepcopy(prefix_cache)  # the cache is extended in place by each query
        if decoding == "generation":
            output = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), past_key_values=cache,
                                    max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                                    temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
            raws.append(tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True))
            preds.append(parse_answer(raws[-1], lang))
        else:
            ranked = sorted(zip(score_labels(model, input_ids, labels_ids, cache), SUPERSENSES), reverse=True)
            raws.append(" | ".join(f"{label} {score:.2f}" for score, label in ranked[:3]))
            preds.append(ranked[0][1])
    return preds, raws

In [7]:
def config_id(lang, n_shots, shot_selection, with_lemma, decoding):
    """Readable run name, e.g. en_25shot_by_class_lemma (decoding added when it is not generation)."""
    shots = f"{n_shots}shot" + (f"_{shot_selection}" if n_shots else "")
    return f"{lang}_{shots}_{'lemma' if with_lemma else 'nolemma'}" + ("_logprob" if decoding == "logprob" else "")


def run(lang, n_shots, shot_selection="by_class", with_lemma=True, decoding="generation", split=CONFIG["tuning_split"]):
    """Predict, score, save and log one configuration; reuse the saved predictions if it was already run."""
    cid = config_id(lang, n_shots, shot_selection, with_lemma, decoding)
    path = RESULTS_DIR / "predictions" / f"icl__{cid}__{split}.csv"
    fresh = not path.exists()
    if not fresh:  # Colab sessions are cut regularly: never pay twice for the same run
        preds = pd.read_csv(path, keep_default_na=False).replace("", None)
    else:
        eval_df = get_split(df, split)
        start = time.time()
        selector = ShotSelector(train, n_shots, shot_selection, CONFIG["seed"])
        pred_labels, raws = predict(eval_df, selector, lang, with_lemma, decoding)
        runtime = (time.time() - start) / 60
        preds = eval_df[["sense_id", "lemma", "definition"]].assign(
            gold_supersense=eval_df["supersense"], pred_supersense=pred_labels,
            gold_hypersense=eval_df["hypersense"], raw_output=raws)
        preds.insert(6, "pred_hypersense", preds["pred_supersense"].map(SUPER2HYPER))
        save_predictions(preds, "icl", cid, split)
    sup = evaluate(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES)
    hyp = evaluate(preds["gold_hypersense"].tolist(), preds["pred_hypersense"].tolist(), HYPERSENSES)
    if fresh:
        log_experiment({"author": "A", "method": "icl", "config_id": cid, "split": split, "model": CONFIG["model"],
                        "with_lemma": with_lemma, "prompt_lang": lang, "n_shots": n_shots,
                        "shot_selection": shot_selection if n_shots else "", "decoding": decoding,
                        "accuracy_super": round(sup["accuracy"], 4), "macro_f1_super": round(sup["macro_f1"], 4),
                        "accuracy_hyper": round(hyp["accuracy"], 4), "macro_f1_hyper": round(hyp["macro_f1"], 4),
                        "invalid_rate": round(sup["invalid_rate"], 4), "n": sup["n"], "runtime_min": round(runtime, 1)})
    print(f"{cid:35s} macro-F1 {sup['macro_f1']:.3f}  acc {sup['accuracy']:.3f}  "
          f"hyper macro-F1 {hyp['macro_f1']:.3f}  invalid {sup['invalid_rate']:.1%}")
    return {"config_id": cid, "macro_f1": sup["macro_f1"], "accuracy": sup["accuracy"],
            "hyper_macro_f1": hyp["macro_f1"], "invalid_rate": sup["invalid_rate"], "preds": preds}


def compare(a, b):
    """Print the share of bootstrap resamples of the evaluation set where b has a higher macro-F1 than a.

    A share close to 100 % or 0 % means a consistent difference; around 50 % means no difference.
    """
    share = paired_bootstrap(a["preds"]["gold_supersense"], a["preds"]["pred_supersense"],
                             b["preds"]["pred_supersense"], SUPERSENSES)
    print(f"{b['config_id']} beats {a['config_id']} in {share:.0%} of bootstrap resamples")

## 1. Prompt language

English instruction and English labels vs French instruction and French labels (inputs are French in both cases).
Compared without demonstrations (the instruction is all the model sees) and with 5 balanced demonstrations.

In [8]:
lang_runs = {(lang, n): run(lang, n) for lang in ["en", "fr"] for n in [0, 5]}
compare(lang_runs["en", 0], lang_runs["fr", 0])
compare(lang_runs["en", 5], lang_runs["fr", 5])
LANG = max(["en", "fr"], key=lambda l: lang_runs[l, 0]["macro_f1"] + lang_runs[l, 5]["macro_f1"])
print("Prompt language kept:", LANG)

en_0shot_lemma                      macro-F1 0.412  acc 0.518  hyper macro-F1 0.479  invalid 0.5%
en_5shot_by_class_lemma             macro-F1 0.442  acc 0.556  hyper macro-F1 0.479  invalid 2.6%
fr_0shot_lemma                      macro-F1 0.377  acc 0.446  hyper macro-F1 0.448  invalid 0.7%
fr_5shot_by_class_lemma             macro-F1 0.364  acc 0.461  hyper macro-F1 0.421  invalid 4.9%
fr_0shot_lemma beats en_0shot_lemma in 8% of bootstrap resamples
fr_5shot_by_class_lemma beats en_5shot_by_class_lemma in 0% of bootstrap resamples
Prompt language kept: en


## 2. Number of demonstrations

Balanced demonstrations (`by_class`): 25 shots = one per class, 50 = two per class.

In [9]:
shot_runs = {n: run(LANG, n) for n in [0, 5, 25, 50]}
N_SHOTS = max(shot_runs, key=lambda n: shot_runs[n]["macro_f1"])
compare(shot_runs[0], shot_runs[N_SHOTS])
print("Number of shots kept:", N_SHOTS)

en_0shot_lemma                      macro-F1 0.412  acc 0.518  hyper macro-F1 0.479  invalid 0.5%
en_5shot_by_class_lemma             macro-F1 0.442  acc 0.556  hyper macro-F1 0.479  invalid 2.6%
en_25shot_by_class_lemma            macro-F1 0.421  acc 0.549  hyper macro-F1 0.426  invalid 5.4%
en_50shot_by_class_lemma            macro-F1 0.434  acc 0.481  hyper macro-F1 0.444  invalid 4.9%
en_5shot_by_class_lemma beats en_0shot_lemma in 89% of bootstrap resamples
Number of shots kept: 5


## 3. Choice of the demonstrations

`random`: a random sample of train (follows the train label distribution);
`by_class`: balanced over labels; `similar`: for each query, the training definitions most similar to its definition
(TF-IDF on character n-grams).

In [10]:
selection_runs = {s: run(LANG, N_SHOTS, s) for s in ["random", "by_class", "similar"]}
SELECTION = max(selection_runs, key=lambda s: selection_runs[s]["macro_f1"])
for s in selection_runs:
    if s != SELECTION:
        compare(selection_runs[s], selection_runs[SELECTION])
print("Selection kept:", SELECTION)

en_5shot_random_lemma               macro-F1 0.406  acc 0.568  hyper macro-F1 0.401  invalid 3.0%
en_5shot_by_class_lemma             macro-F1 0.442  acc 0.556  hyper macro-F1 0.479  invalid 2.6%
en_5shot_similar_lemma              macro-F1 0.476  acc 0.625  hyper macro-F1 0.471  invalid 1.8%
en_5shot_similar_lemma beats en_5shot_random_lemma in 99% of bootstrap resamples
en_5shot_similar_lemma beats en_5shot_by_class_lemma in 89% of bootstrap resamples
Selection kept: similar


## 4. Is the lemma useful?

In [11]:
lemma_runs = {w: run(LANG, N_SHOTS, SELECTION, with_lemma=w) for w in [True, False]}
compare(lemma_runs[False], lemma_runs[True])
WITH_LEMMA = lemma_runs[True]["macro_f1"] >= lemma_runs[False]["macro_f1"]

en_5shot_similar_lemma              macro-F1 0.476  acc 0.625  hyper macro-F1 0.471  invalid 1.8%
en_5shot_similar_nolemma            macro-F1 0.502  acc 0.661  hyper macro-F1 0.531  invalid 1.9%
en_5shot_similar_lemma beats en_5shot_similar_nolemma in 11% of bootstrap resamples


## 5. Generation vs label scoring

Same prompt as the best configuration so far; the model scores the 25 labels instead of generating.

In [12]:
best_gen = lemma_runs[WITH_LEMMA]
decoding_runs = {"generation": best_gen, "logprob": run(LANG, N_SHOTS, SELECTION, WITH_LEMMA, decoding="logprob")}
compare(decoding_runs["generation"], decoding_runs["logprob"])
DECODING = max(decoding_runs, key=lambda d: decoding_runs[d]["macro_f1"])
print("Decoding kept:", DECODING)

en_5shot_similar_nolemma_logprob    macro-F1 0.509  acc 0.675  hyper macro-F1 0.532  invalid 0.0%
en_5shot_similar_nolemma_logprob beats en_5shot_similar_nolemma in 84% of bootstrap resamples
Decoding kept: logprob


## Summary of the tuning runs

In [13]:
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
metrics[(metrics["method"] == "icl") & (metrics["split"] == CONFIG["tuning_split"])].drop_duplicates(
    "config_id", keep="last").sort_values("macro_f1_super", ascending=False)[
    ["config_id", "macro_f1_super", "accuracy_super", "macro_f1_hyper", "invalid_rate", "runtime_min"]]

,config_id,macro_f1_super,accuracy_super,macro_f1_hyper,invalid_rate,runtime_min
9,en_5shot_similar_nolemma_logprob,0.5091,0.6754,0.5325,0.0000,25.6
8,en_5shot_similar_nolemma,0.5022,0.6614,0.5312,0.0193,16.8
7,en_5shot_similar_lemma,0.4757,0.6246,0.4711,0.0175,18.9
1,en_5shot_by_class_lemma,0.4419,0.5561,0.4792,0.0263,7.0
5,en_50shot_by_class_lemma,0.4342,0.4807,0.4440,0.0491,7.7
4,en_25shot_by_class_lemma,0.4208,0.5491,0.4258,0.0544,7.4
0,en_0shot_lemma,0.4120,0.5175,0.4787,0.0053,6.9
6,en_5shot_random_lemma,0.4056,0.5684,0.4010,0.0298,7.0
2,fr_0shot_lemma,0.3771,0.4456,0.4480,0.0070,8.3
3,fr_5shot_by_class_lemma,0.3641,0.4614,0.4213,0.0491,8.0


## Conclusions on rand-dev

- English instruction and labels work better than French ones for French inputs (0 shots: 0.412 vs 0.377 macro-F1;
  5 shots: 0.442 vs 0.364), as reported by Lai et al. (2023). With French labels the model also invents more labels.
- Five balanced demonstrations help (0.442 vs 0.412 without, better in 89 % of bootstrap resamples), but 25 or 50
  do not help more (0.421, 0.434): longer prompts make the model invent labels outside the list (5 % invalid answers).
- The choice of the demonstrations matters more than their number: the 5 training definitions most similar to the
  query (0.476) beat random (0.406) and balanced (0.442) demonstrations, in 99 % and 89 % of resamples.
- The lemma does not help the prompted model: without it macro-F1 is 0.502 vs 0.476 (better in 89 % of resamples);
  the lemma fixes 24 senses and breaks 45, often rare or multiword lemmas that mislead the model
  (see `01b_icl_error_analysis`).
- Scoring the 25 labels instead of generating gives 0.509 vs 0.502 (better in 84 % of resamples) and no invalid
  answer, at the cost of a longer runtime.

Final configuration: English prompt, 5 similar demonstrations, no lemma, label scoring.

## Final evaluation on the test sets

The configuration chosen on `rand-dev` above is run **once** on each test set; nothing is tuned after this point.

In [14]:
test_runs = {split: run(LANG, N_SHOTS, SELECTION, WITH_LEMMA, DECODING, split=split) for split in ["rand-test", "freq-test"]}

en_5shot_similar_nolemma_logprob    macro-F1 0.583  acc 0.717  hyper macro-F1 0.569  invalid 0.0%
en_5shot_similar_nolemma_logprob    macro-F1 0.519  acc 0.523  hyper macro-F1 0.536  invalid 0.0%


On the test sets the final configuration reaches a macro-F1 of 0.583 on `rand-test` (accuracy 0.717) and 0.519
on `freq-test` (accuracy 0.523). The gap in accuracy comes mostly from the composition of the splits: without its
113 demonyms, `rand-test` accuracy is 0.66, while `freq-test` has a single demonym and many senses of frequent,
polysemous nouns (`act` vs `act*cognition` or `event`, `artifact` vs `object`). See `01b_icl_error_analysis`.

## Download the results

On Colab, the prediction files and log rows written by this session are zipped and downloaded, so that they can be added to the repository. On Kaggle, everything written under `/kaggle/working` is kept as the notebook's output.

In [15]:
if "google.colab" in sys.modules:
    from google.colab import files
    !cd /content/LAB1_classification_with_llms && zip -qr /content/icl_results.zip results
    files.download("/content/icl_results.zip")